In [ ]:
# ============================================================
# CELDA 1 — EXTRACCIÓN DE NOMENCLATURA DESDE ARCHIVOS .fif
# ============================================================

from pathlib import Path
import pandas as pd
import re

# Carpeta con los .fif preprocesados
FIF_DIR = Path(r"C:\Users\pokem\OneDrive\Desktop\lab 2026\EEG_Preprocesado_GoNoGo")

# IDs de adultos (30 sujetos — fuente: carpeta ADULTOS/Sin Contexto del desktop)
ADULTS = {
    'AAELSC', 'AAGFSC',  'EDGSSIN',  'FIJRBSIN',
    'GSASC',  'IJALSC',  'JAGSSC',   'LHACSIN',
    'MAEGLSC','MAHLSC',  'MFGSIN',   'MGOKSC',
    'RAMPSIN','SDBGSIN', 'VMRSIN',   'AVLLSC',
    'BMPSC',  'CGMSIN',  'CJGSIN',   'CPGSC',
    'CPMHSIN','GGMSIN',  'IABWSC',   'JAUSIN',
    'JGCSC',  'MGVGSC',  'RKACSC',   'SEOOSIN',
    'TTOSC',  'VBPSIN',
}

# Buscar todos los archivos -epo.fif
fif_files = sorted(FIF_DIR.glob('*-epo.fif'))
print(f"Archivos encontrados: {len(fif_files)}")
print()

rows = []
for fpath in fif_files:
    stem = fpath.stem          # AAELSCGO_GO_clean-epo
    name = stem.replace('-epo', '')  # AAELSCGO_GO_clean

    # Condición: token explícito _GO_ o _NOGO_ entre guiones bajos
    match = re.search(r'_(NOGO|GO)_', name, re.IGNORECASE)
    condition = match.group(1).upper() if match else 'UNKNOWN'

    # ID del sujeto (parte antes del token de condición)
    subject_id = re.split(r'_(?:NOGO|GO)_', name, flags=re.IGNORECASE)[0]

    # Base del sujeto sin sufijo de condición (CGMSINGO → CGMSIN)
    sid = subject_id.upper()
    if sid.endswith('NOGO'):    base_sub = sid[:-4]
    elif sid.endswith('SINGO'): base_sub = sid[:-2]
    elif sid.endswith('NGO'):   base_sub = sid[:-3]
    elif sid.endswith('NG'):    base_sub = sid[:-2]
    elif sid.endswith('GO'):    base_sub = sid[:-2]
    else:                       base_sub = sid

    group = 'ADULTO' if base_sub in ADULTS else 'ADOLESCENTE'

    rows.append({
        'archivo':    fpath.name,
        'subject_id': subject_id,
        'condition':  condition,
        'group':      group,
        'features_key': f"{subject_id}_{condition}",
    })

df_labels = pd.DataFrame(rows)

print("DISTRIBUCIÓN:")
print(df_labels.groupby(['group', 'condition']).size().to_string())
print()
print("UNKNOWN (revisar):")
unknowns = df_labels[df_labels.apply(
    lambda r: r['condition'] == 'UNKNOWN' or r['group'] == 'UNKNOWN', axis=1
)]
if len(unknowns):
    print(unknowns[['archivo', 'subject_id']].to_string())
else:
    print("  ✅ Ninguno")
print()
print("Vista previa:")
display(df_labels.head(10))

In [2]:
# ============================================================
# CELDA 2 — CRUCE CON features_dir Y VALIDACIÓN
# ============================================================
# Pega PROJECT_ROOT y features_dir de tu notebook de análisis

from pathlib import Path
import numpy as np

FEATURES_DIR = Path(r"C:\Proyectos\eeg_hmm_plattform\data\interim\features\task_hjorthonly_temporalqc")

# Construir mapa: stem del .npy → metadatos
npy_files = sorted(FEATURES_DIR.glob('*_features.npy'))
print(f"Archivos .npy encontrados: {len(npy_files)}")
print()

# Muestra los primeros 5 para ver el formato exacto
print("Primeros 5 nombres de .npy:")
for f in npy_files[:5]:
    print(f"  {f.stem}")
print()

# Construir lookup desde df_labels
# Intenta cruzar por subject_id + condition con el stem del .npy
label_lookup = {}
for _, row in df_labels.iterrows():
    # Patrones posibles en el .npy:
    # AAELSCGO_GO_sin_contexto_features
    # AAELSCGO_sin_contexto_features  (sin condición explícita)
    key1 = f"{row['subject_id']}_GO"    # para GO
    key2 = f"{row['subject_id']}_NOGO"  # para NOGO
    key3 = row['subject_id']             # sin condición
    label_lookup[row['subject_id'].upper()] = row.to_dict()

# Cruzar cada .npy con sus metadatos
matched, unmatched = [], []

for fpath in npy_files:
    stem = fpath.stem.upper()  # AAELSCGO_SIN_CONTEXTO_FEATURES
    
    # Detectar condición desde el nombre del .npy
    if '_NOGO_' in stem or stem.endswith('NG_SIN_CONTEXTO_FEATURES'):
        cond_npy = 'NOGO'
    elif '_GO_' in stem and 'NOGO' not in stem:
        cond_npy = 'GO'
    else:
        # Intentar detectar por sufijo del ID (GO/NG al final del ID)
        parts = fpath.stem.split('_')
        first_part = parts[0].upper()
        cond_npy = ('NOGO' if first_part.endswith('NG') else
                    'GO'   if first_part.endswith('GO') else
                    'UNKNOWN')
    
    # Buscar en df_labels
    match = df_labels[
        (df_labels['condition'] == cond_npy) &
        df_labels['subject_id'].apply(
            lambda sid: sid.upper() in fpath.stem.upper()
        )
    ]
    
    if len(match) == 1:
        row = match.iloc[0]
        matched.append({
            'npy_file':   fpath.name,
            'subject_id': row['subject_id'],
            'condition':  row['condition'],
            'group':      row['group'],
        })
    elif len(match) > 1:
        unmatched.append({'npy_file': fpath.name, 'motivo': 'múltiples matches'})
    else:
        unmatched.append({'npy_file': fpath.name, 'motivo': f'sin match (cond={cond_npy})'})

df_matched   = pd.DataFrame(matched)
df_unmatched = pd.DataFrame(unmatched)

print(f"✅ Cruzados exitosamente: {len(matched)}/{len(npy_files)}")
print(f"⚠️  Sin cruzar:          {len(unmatched)}/{len(npy_files)}")
print()

if len(df_unmatched):
    print("SIN CRUZAR — revisar manualmente:")
    display(df_unmatched)
    print()

print("DISTRIBUCIÓN FINAL (archivos .npy):")
print(df_matched.groupby(['group','condition']).size().to_string())
print()

# Exportar el mapa como CSV para usar en el notebook de análisis
OUT_CSV = Path(r"C:\Proyectos\eeg_hmm_plattform\configs\subject_labels.csv")
df_matched.to_csv(OUT_CSV, index=False)
print(f"✅ Labels exportados: {OUT_CSV}")
print()
print("Para usar en deep_analysis.ipynb, carga así:")
print(f"  df_labels_ext = pd.read_csv(r'{OUT_CSV}')")
print(f"  # Luego cruza por 'npy_file' con los feature_files")

Archivos .npy encontrados: 118

Primeros 5 nombres de .npy:
  AAELSCGO_sin_contexto_features
  AAELSCNG_sin_contexto_features
  AAGFSCGO_sin_contexto_features
  AAGFSCNG_sin_contexto_features
  ACBSINGO_sin_contexto_features

✅ Cruzados exitosamente: 118/118
⚠️  Sin cruzar:          0/118

DISTRIBUCIÓN FINAL (archivos .npy):
group        condition
ADOLESCENTE  GO           28
             NOGO         28
ADULTO       GO           16
             NOGO         16
UNKNOWN      GO           15
             NOGO         15

✅ Labels exportados: C:\Proyectos\eeg_hmm_plattform\configs\subject_labels.csv

Para usar en deep_analysis.ipynb, carga así:
  df_labels_ext = pd.read_csv(r'C:\Proyectos\eeg_hmm_plattform\configs\subject_labels.csv')
  # Luego cruza por 'npy_file' con los feature_files
